# 5. Delta Sharing – Compartir datos con terceros
Se comparten las tablas de negocio con un socio externo (red de concesionarios) sin copiar datos, usando **Delta Sharing** (protocolo abierto).

| Objeto | Nombre |
|---|---|
| Share | `VEHICULOS_SHARE` |
| Recipient | `PartnerConcesionarios` (open sharing: recibe un enlace de activación) |
| Tablas | 4 tablas golden + `silver.car_features_clean` |

> No se comparte `silver.car_sales_enriched` porque contiene datos personales de clientes (nombre, teléfono, ingreso).
> Requisitos: Delta Sharing habilitado en el metastore y privilegios `CREATE SHARE` / `CREATE RECIPIENT`.

### Datos a compartir

In [0]:
SELECT * FROM vehiculos.golden.kpi_concesionarios ORDER BY monto_total DESC

### Creación del share
Se elimina primero para que el notebook sea re-ejecutable.

In [0]:
DROP SHARE IF EXISTS VEHICULOS_SHARE;

In [0]:
CREATE SHARE IF NOT EXISTS VEHICULOS_SHARE
COMMENT 'Indicadores de venta de vehiculos para socios externos';

In [0]:
SHOW SHARES;

### Agregar tablas al share
`WITH HISTORY` permite al receptor consultar versiones anteriores (time travel) y leer en streaming.

In [0]:
ALTER SHARE VEHICULOS_SHARE
ADD TABLE vehiculos.golden.ventas_mensuales_estado
WITH HISTORY;

In [0]:
ALTER SHARE VEHICULOS_SHARE
ADD TABLE vehiculos.golden.ranking_modelos
WITH HISTORY;

In [0]:
ALTER SHARE VEHICULOS_SHARE
ADD TABLE vehiculos.golden.kpi_concesionarios
WITH HISTORY;

In [0]:
ALTER SHARE VEHICULOS_SHARE
ADD TABLE vehiculos.golden.segmentos_clientes
WITH HISTORY;

In [0]:
ALTER SHARE VEHICULOS_SHARE
ADD TABLE vehiculos.silver.car_features_clean
WITH HISTORY;

In [0]:
SHOW ALL IN SHARE VEHICULOS_SHARE;

### Creación del recipient

In [0]:
DROP RECIPIENT IF EXISTS PartnerConcesionarios;

In [0]:
CREATE RECIPIENT IF NOT EXISTS PartnerConcesionarios
COMMENT 'Red de concesionarios externa';

El campo `activation_link` es el enlace que se envía al socio para descargar su credencial (archivo `.share`).

In [0]:
DESCRIBE RECIPIENT PartnerConcesionarios;

### Otorgar acceso

In [0]:
GRANT SELECT
ON SHARE VEHICULOS_SHARE
TO RECIPIENT PartnerConcesionarios;

In [0]:
SHOW GRANTS ON SHARE VEHICULOS_SHARE;

In [0]:
SHOW GRANTS TO RECIPIENT PartnerConcesionarios;

### Historial compartido
Cada ejecución del workflow genera una nueva versión de las tablas golden; el receptor puede consultarlas gracias a `WITH HISTORY`.

In [0]:
DESCRIBE HISTORY vehiculos.golden.kpi_concesionarios;

### Revocar acceso (opcional)

In [0]:
-- REVOKE SELECT ON SHARE VEHICULOS_SHARE FROM RECIPIENT PartnerConcesionarios;